# 2.3 Function Calling - ToolRegistry

构建Agent的工具插件系统, 支持动态注册、安全执行和MCP兼容接口

## 项目定位

本实验的 ToolRegistry 将被4.1 ReActAgent和6.3最终项目使用

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client, verify_config
from agent_project import *
from agent_project.tools import create_default_registry

client = get_client()
print(f"项目模块已加载 | LLM: {client.name} ({client.model})")


In [ ]:
# 创建工具注册表
from agent_project.tools import create_default_registry, SafeCalculator

tools = create_default_registry()
print(f"已注册工具: {tools.list_tools()}")
print(f"工具描述:\n{tools.get_description()}")


In [ ]:
# 工具执行测试
print("===== 工具执行测试 =====\n")

tests = [
    ("search", {"query": "MCP"}),
    ("calculator", {"expression": "sqrt(144) + 3 * 5"}),
    ("datetime", {}),
]

for name, args in tests:
    result = tools.execute(name, args)
    print(f"  {name}({args}): {result[:80]}")


In [ ]:
# 注册自定义工具: 知识库统计
def kb_stats():
    """获取知识库统计信息"""
    return "知识库: 50篇文档, 320个文本块, 最后更新2026-06-28"

tools.register("kb_stats", kb_stats, "获取知识库统计信息",
    {"type": "object", "properties": {}, "required": []})
print(f"注册新工具后: {tools.list_tools()}")
print(tools.execute("kb_stats", {}))


In [ ]:
# Function Calling: 让LLM自动选择工具
print("===== LLM驱动的工具选择 =====\n")

queries = [
    "搜索一下MCP协议的定义",
    "计算 25 * 4 + 100 / 5",
    "现在几点?",
    "你好, 今天星期几?",  # 不需要工具
]

for query in queries:
    r = client.chat(
        system="你是智能助手。需要时使用工具获取信息。",
        messages=[{"role": "user", "content": query}],
        tools=tools.get_openai_schemas(),
        temperature=0.1, max_tokens=200
    )
    if r.get("tool_calls"):
        tc = r["tool_calls"][0]
        result = tools.execute(tc["name"], tc["arguments"])
        print(f"  [{tc['name']}] {query} -> {result[:60]}")
    else:
        print(f"  [直接回答] {query} -> {r['content'][:60]}")
